# Idempotent silver
Bronze keeps every delivery, silver keeps one row per `event_id`. The stream reads only new bronze commits and a micro-batch can be applied more than once with the same result:

- `dropDuplicates` removes repeats inside the batch
- `MERGE` with insert-only semantics ignores events that silver already holds, including a micro-batch repeated after a failure

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

from delta.tables import DeltaTable

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"
bronze = f"{target}.events_bronze"
silver = f"{target}.events_silver"

In [ ]:
def merge_batch(batch, batch_id):
    (
        DeltaTable.forName(batch.sparkSession, silver).alias("t")
        .merge(batch.dropDuplicates(["event_id"]).alias("s"), "t.event_id = s.event_id")
        .whenNotMatchedInsertAll()
        .execute()
    )

In [ ]:
silver_before = spark.table(silver).count()

(
    spark.readStream.table(bronze)
    .writeStream
    .foreachBatch(merge_batch)
    .option("checkpointLocation", f"/Volumes/{catalog}/{schema}/checkpoints/silver")
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

silver_after = spark.table(silver).count()
print(f"bronze {spark.table(bronze).count()}, silver {silver_before} -> {silver_after}")